# CME Futures: Principal-Component Factors

PCA compresses the point-in-time feature panel into fold-scoped components. The transformer fits
on training rows only, and the saved fitted state is reused to transform that fold's validation
rows. Both return horizons are declared explicitly.

This notebook publishes predictions and fitted-state lineage. `13_backtest` applies the common
validation-Sharpe selection rule.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Fit the declared CME futures PCA factor population."""

import polars as pl

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    model_request_catalog,
    open_study,
    product_universe_table,
    resolve_model_requests,
    resolved_model_plan,
    run_official_model_catalog,
    run_resolved_model_requests,
)

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_REDUCTIONS: dict = {}

## Declared requests

Both return horizons use the named PCA configuration. The resolved plan shows the eligible rows,
folds, feature count, checkpoint schedule, and identity before fitting begins.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
requests = model_request_catalog(
    "latent_factors",
    labels=ALL_LABELS,
    config_names=("pca",),
)
resolved = resolve_model_requests(
    study,
    requests,
    execution_tier=EXECUTION_TIER,
    preview_reductions=PREVIEW_REDUCTIONS,
)
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


In [4]:
resolved_model_plan(resolved)

family,label,config_name,task,feature_count,eligible_entities,eligible_rows,folds,validation_start,validation_end,checkpoints,execution_tier,training_hash
str,str,str,str,i64,i64,i64,i64,datetime[μs],datetime[μs],i64,str,str
"""latent_factors""","""fwd_ret_21d""","""pca""","""regression""",69,30,37008,5,2019-01-03 00:00:00,2023-11-29 00:00:00,1,"""canonical""","""164ce7df51e5"""
"""latent_factors""","""fwd_ret_5d""","""pca""","""regression""",69,30,37488,5,2019-01-03 00:00:00,2023-12-21 00:00:00,1,"""canonical""","""4a8088d47213"""


## Execute and validate

The shared latent-factor runner fits PCA inside each training fold, persists the transformer, and
requires the complete validation key set before publication.

In [5]:
if EXECUTION_TIER == "canonical":
    execution, population = run_official_model_catalog(
        study,
        requests,
        population_name="cme-pca-validation-v1",
        resolved_requests=resolved,
    )
else:
    if WORKSPACE is None or not PREVIEW_REDUCTIONS:
        raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
    execution = run_resolved_model_requests(study, resolved)
    population = None

Latent factor CV: 1 models × 5 folds
Log file: case_studies/cme_futures/run_log/training/4a8088d47213/.models.05e4de49528745c0ab4f2b34c62080e7.tmp/latent_factors.log
Scoring: dates=all cadence=- step=1 checkpoint_selection=fixed reporting_epoch=0
  pca (K=5):
    Fold 0: persistent train=2059, val=253, max_N=30


      fold 0: reported_epoch=0, IC=-0.0894, 0.1s
    Fold 1: persistent train=2059, val=258, max_N=30


      fold 1: reported_epoch=0, IC=-0.0370, 0.0s
    Fold 2: persistent train=2059, val=258, max_N=29


      fold 2: reported_epoch=0, IC=-0.0066, 0.0s
    Fold 3: persistent train=2059, val=258, max_N=29


      fold 3: reported_epoch=0, IC=-0.0356, 0.0s
    Fold 4: persistent train=2039, val=258, max_N=29


...

      fold 1: reported_epoch=0, IC=-0.0926, 0.0s
    Fold 2: persistent train=2043, val=258, max_N=29


      fold 2: reported_epoch=0, IC=-0.1139, 0.0s
    Fold 3: persistent train=2043, val=258, max_N=29


      fold 3: reported_epoch=0, IC=-0.1118, 0.0s
    Fold 4: persistent train=2022, val=258, max_N=29


      fold 4: reported_epoch=0, IC=+0.1590, 0.0s
    -> best epoch=0, IC=-0.0015 (1.2s)


  Best: pca (IC=-0.0015)


In [6]:
catalog = execution.catalog_rows.select(
    "family",
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "execution_tier",
    "complete",
    "training_hash",
    "prediction_hash",
).sort("label", "checkpoint_value")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("PCA execution returned a partial prediction")
catalog

family,label,config_name,checkpoint_kind,checkpoint_value,execution_tier,complete,training_hash,prediction_hash
str,str,str,str,i64,str,bool,str,str
"""latent_factors""","""fwd_ret_21d""","""pca""","""epoch""",0,"""canonical""",true,"""164ce7df51e5""","""863984af3db2"""
"""latent_factors""","""fwd_ret_5d""","""pca""","""epoch""",0,"""canonical""",true,"""4a8088d47213""","""187857c9b69d"""
